In [1]:
import csv
import json

from scielo_scholarly_data import standardizer

In [2]:
base_to_keys = {
    'doaj': {
        'titles': ['Journal title', 'Alternative title'],
        'countries': ['Country of publisher', 'Country of other organisation'],
        'issns': ['Journal ISSN (print version)', 'Journal EISSN (online version)'],
        'delimiter': ',',
    }, 
    'latindex': {
        'titles': ['tit_propio',],
        'countries': ['nombre_largo'],
        'issns': ['issn_e', 'issn_l', 'issn_imp'],
        'delimiter': ';',
    },
    'ms_brazil': {
        'titles': ['"Título"', '"Outro Título"'],
        'countries': ['"País"',],
        'issns': ['"ISSN"',],
        'issn_list': ['"Outro(s) ISSN(s)"'],
        'issn_list_delimiter': '',
        'delimiter': ',',
        'quoting': csv.QUOTE_ALL,
    },
    'ms_spain': {
        'titles': ['"Título"', '"Outro Título"'],
        'countries': ['"País"',],
        'issns': ['"ISSN"',],
        'issn_list': ['"Outro(s) ISSN(s)"'],
        'issn_list_delimiter': '',
        'delimiter': ',',
        'quoting': csv.QUOTE_ALL,
    },
    'scielo': {
        'titles': ['"title at SciELO"', '"title + subtitle SciELO"', '"short title SciELO"', '"short title ISO"', '"title PubMed"'],
        'countries': ['"collection"'],
        'issns': ['"ISSN SciELO"'],
        'issn_list': ['"ISSN\'s"',],
        'issn_list_delimiter': ';',
        'delimiter': ',',
        'quoting': csv.QUOTE_ALL,
    },
    'scimagojr': {
        'titles': ['Title',],
        'countries': ['Country'],
        'issns': [],
        'issn_list': ['Issn',],
        'issn_list_delimiter': ', ',
        'delimiter': ';'
    },
    'scopus_sources': {
        'titles': ['Source Title', 'Medline-sourced Title? (See additional details under separate tab.)', 'Related title 1', 'Other related title 2', 'Other related title 3', 'Other related title 4'],
        'countries': [],
        'issns': ['Print-ISSN', 'E-ISSN'],
        'delimiter': ','
    },
    'scopus_accepted': {
        'titles': ['Title name'],
        'countries': [],
        'issns': ['Print ISSN', 'EISSN'],
        'delimiter': ','
    },
    'ulrich': {
        'titles': ['"title_title"'],
        'countries': ['"title_country"'],
        'issns': ['"title_issn"'],
        'delimiter': ',',
        "quoting": csv.QUOTE_ALL,
    },
    'wos_extra': {
        'titles': ['"Title"'],
        'countries': ['"Country / Region"'],
        'issns': ['"uISSN"',],
        'issn_list': ['"ISSN/e-ISSN"'],
        'issn_list_delimiter': ' / ',
        'delimiter': ','
    },
    'wos_jcr': {
        'titles': ['title', 'title20'],
        'countries': ['country'],
        'issns': ['eISSN', 'ISSN'],
        'delimiter': ',',
    },
    'mi_extra': {
        'titles': ['title 1', 'title 2'],
        'countries': [],
        'issns': ['issn',],
        'delimiter': '|',
    },
    # special cases
    'nlm': {
        'titles': ['Title Abbreviation', 'Title(s)'],
        'countries': ['Country of Publication'],
        'issns': ['ISSN',],
        'delimiter': '',
    },
    'portal_issn_2024': {
        'main_title': '"main_title"',
        'titles': ['"key_title"',],
        'countries': ['"country"'],
        'issns': ['"issn_l"', '"issn"'],
        'delimiter': ',',
    },
    'portal_issn_2019': {
        'main_title': 'title_proper',
        'main_abbreviated_title': 'abbreviated_key_title',
        'title_list': ['key_title', 'title_proper', 'abbreviated_key_title', 'caption_title', 'other_variant_title'],
        'country_list': ['country'],
        'issn_list': ['issn', 'issn_l'],
        'delimiter': ',',
    },
}

def stz_data(titles, countries, issns):
    titles = list(set([t for t in [standardizer.journal_title_for_deduplication(t).upper() for t in titles] if t != '']))
    countries = list(set([c for c in [standardizer.document_title_for_deduplication(c).upper() for c in countries] if c != '']))
    issns = list(set([i for i in [standardizer.journal_issn(i) for i in issns if i != ''] if i is not None]))
    return {'titles': titles, 'countries': countries, 'issns': issns}

def load_base(path, name, debug=False, limit=False):
    title_keys = base_to_keys[name]['titles']
    country_keys = base_to_keys[name]['countries']
    issn_keys = base_to_keys[name]['issns']
    issn_list_keys = base_to_keys[name].get('issn_list')
    issn_list_delimiter = base_to_keys[name].get('issn_list_delimiter')
    delimiter = base_to_keys[name]['delimiter']
    quoting = base_to_keys[name].get('quoting', csv.QUOTE_MINIMAL)

    data = []
    if debug:
        issns_len = {}
        issn_list_len = 0

    if limit:
        counter = 100

    with open(path) as fin:
        for row in csv.DictReader(fin, fieldnames=fin.readline().strip().split(delimiter), delimiter=delimiter, quoting=quoting):
            if limit:
                counter -= 1
                if counter == 0:
                    break
        
            titles = [row[t] for t in title_keys]
            countries = [row[c] for c in country_keys]
            issns = [row[i] for i in issn_keys]

            if debug:
                for ik in issn_keys:
                    if ik not in issns_len:
                        issns_len[ik] = 0

                    if len(row[ik]) > issns_len[ik]:
                        issns_len[ik] = len(row[ik])
                        print(f'DEBUG. Field issns {ik} contains {issns_len[ik]} chars.')

            if issn_list_keys is not None and issn_list_delimiter is not None:
                if issn_list_delimiter != '':
                    for k in issn_list_keys:
                        for i in row[k].split(issn_list_delimiter):
                            issns.append(i)
                        
                        if debug:
                            if issn_list_len < len(row[k]):
                                issn_list_len = len(row[k])
                                print(f'DEBUG. Field issn_list contains {issn_list_len} chars.')
                else:
                    for k in issn_list_keys:
                        k_size = len(row[k])
                        for ks in range(0, k_size, 9):
                            issns.append(row[k][ks:ks+9])

            norm_data = stz_data(titles, countries, issns)

            if name == 'portal_issn_2024':
                main_title_key = base_to_keys['portal_issn_2024']['main_title']
                norm_data.update({'main_title': standardizer.journal_title_for_deduplication(row.get(main_title_key)).upper()})
            
            data.append(norm_data)
            
    return data

def load_base_nlm(path):
    data = []

    def parse_entry(entry):
        titles = []
        countries = []
        issns = []

        for i in entry.split('\n'):
            if i.startswith('Title Abbr'):
                titles.append(i.split(' Title Abbreviation: ')[-1])

            if i.startswith('Title(s)'):
                titles.append(i.split('Title(s): ')[-1])

            if i.startswith('Country o'):
                countries.append(i.split('Country of Publication: ')[-1])

            if i.startswith('ISSN'):
                temp = [x.strip() for x in i.split('ISSN:')]
                if len(temp) > 0:
                    for x in temp:
                        for xi in x.split('; '):
                            issns.append(xi[0:9])

        return stz_data(titles, countries, issns)

    def parse_text(text):
        entries = text.strip().split('\n\n')
        return [parse_entry(entry) for entry in entries]

    def read_file(file_path):
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()

    text = read_file(path)
    parsed_data = parse_text(text)
    for entry in parsed_data:
        data.append(entry)

    return data

def load_base_portal_issn_2019(path, limit=False):
    data = []

    title_keys = base_to_keys['portal_issn_2019']['title_list']
    country_keys = base_to_keys['portal_issn_2019']['country_list']
    issn_keys = base_to_keys['portal_issn_2019']['issn_list']
    main_title_key = base_to_keys['portal_issn_2019']['main_title']
    main_abbreviated_title_key = base_to_keys['portal_issn_2019']['main_abbreviated_title']

    if limit:
        counter = 100

    for line in open(path):
        if limit:
            counter -= 1
            if counter == 0:
                break
        row = json.loads(line)
        titles = []
        countries = []
        issns = []

        for tk in title_keys:
            titles.extend(row.get(tk, []))

        for ck in country_keys:
            countries.extend(row.get(ck, []))

        for ik in issn_keys:
            issns.extend(row.get(ik, []))
            
        norm_data = stz_data(titles, countries, issns)
        norm_data.update({
            'main_title': set([standardizer.journal_title_for_deduplication(x).upper() for x in row.get(main_title_key, '') if x is not None]), 
            'main_abbreviated_title': set([standardizer.journal_title_for_deduplication(x).upper() for x in row.get(main_abbreviated_title_key, '') if x is not None])
        })

        data.append(norm_data)

    return data

In [3]:
portal_issn_2019 = load_base_portal_issn_2019('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/portal_issn_2019.json')
portal_issn_2019[0:3]

[{'titles': [],
  'countries': [],
  'issns': [],
  'main_title': set(),
  'main_abbreviated_title': set()},
 {'titles': ['PW PUBLISHERS WEEKLY',
   'PW',
   'THE PUBLISHERS WEEKLY',
   'PUBLISHERS WEEKLY',
   'PUBL WKLY'],
  'countries': ['UNITED STATES'],
  'issns': ['0000-0019'],
  'main_title': {'THE PUBLISHERS WEEKLY'},
  'main_abbreviated_title': {'PUBL WKLY'}},
 {'titles': ['LIBRARY JOURNAL 1876',
   'LIBRARY JOURNAL',
   'LIBR J 1876',
   'AMERICAN LIBRARY JOURNAL'],
  'countries': ['UNITED STATES'],
  'issns': ['0000-0027'],
  'main_title': {'LIBRARY JOURNAL'},
  'main_abbreviated_title': {'LIBR J 1876'}}]

In [4]:
portal_issn_2024 = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/portal_issn_2024.csv', 'portal_issn_2024')
portal_issn_2024[0:3]

[{'titles': ['KALTARA SEHAT'],
  'countries': ['INDONESIA'],
  'issns': ['2685-6352'],
  'main_title': 'KALTARA SEHAT'},
 {'titles': ['L ACTION LIBERALE POPULAIRE CLERMONT FERRAND'],
  'countries': ['FRANCE'],
  'issns': ['2120-1412'],
  'main_title': 'L ACTION LIBERALE POPULAIRE'},
 {'titles': ['DEV SOFTWARE SISTEMI & SOLUZIONI TESTO STAMPATO'],
  'countries': ['ITALY'],
  'issns': ['1124-5468'],
  'main_title': 'DEV SOFTWARE SISTEMI & SOLUZIONI'}]

In [5]:
doaj = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/doaj.csv', 'doaj', debug=True)
doaj[0:3]

DEBUG. Field issns Journal EISSN (online version) contains 9 chars.
DEBUG. Field issns Journal ISSN (print version) contains 9 chars.


[{'titles': ['COMUNICOLOGIA'],
  'countries': ['BRAZIL'],
  'issns': ['1981-2132']},
 {'titles': ['REVISTA DE CIENCIA Y TECNOLOGIA', 'RECYT'],
  'countries': ['ARGENTINA'],
  'issns': ['0329-8922', '1851-7587']},
 {'titles': ['UNIVERSIDAD Y EMPRESA', 'REVISTA UNIVERSIDAD Y EMPRESA'],
  'countries': ['COLOMBIA'],
  'issns': ['0124-4639', '2145-4558']}]

In [7]:
latindex = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/latindex.csv', 'latindex', debug=True)
latindex[0:3]

DEBUG. Field issns issn_imp contains 9 chars.
DEBUG. Field issns issn_e contains 9 chars.
DEBUG. Field issns issn_l contains 9 chars.
DEBUG. Field issns issn_imp contains 10 chars.
DEBUG. Field issns issn_e contains 10 chars.


[{'titles': ['MUCHO AMBIENTE'], 'countries': ['MEXICO'], 'issns': []},
 {'titles': ['VIVA EL TANGO'],
  'countries': ['ARGENTINA'],
  'issns': ['1514-1470']},
 {'titles': ['MAS DERECHO'],
  'countries': ['ARGENTINA'],
  'issns': ['1515-6753']}]

In [8]:
ms_brazil = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/ms_brazil.csv', 'ms_brazil', debug=True)
ms_brazil[0:3]

DEBUG. Field issns "ISSN" contains 9 chars.
DEBUG. Field issns "ISSN" contains 11 chars.


[{'titles': ['CABORE', 'REVISTA CABORE'],
  'countries': ['BRASIL'],
  'issns': ['0007-9316']},
 {'titles': ['REVISTA BRASILEIRA DE OTORRINOLARINGOLOGIA'],
  'countries': ['BRASIL'],
  'issns': ['0034-7299', '1806-9312']},
 {'titles': ['REVISTA DE ADMINISTRACAO PUBLICA'],
  'countries': ['BRASIL'],
  'issns': ['0034-7612', '1982-3134']}]

In [9]:
ms_spain = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/ms_spain.csv', 'ms_spain', debug=True)
ms_spain[0:3]

DEBUG. Field issns "ISSN" contains 9 chars.


[{'titles': ['PANGEAS'], 'countries': ['ESPANHA'], 'issns': ['2695-5040']},
 {'titles': ['REVISTA ESPANOLA DE MEDICINA LEGAL'],
  'countries': ['ESPANHA'],
  'issns': ['0377-4732', '2173-917X']},
 {'titles': ['ATENCION PRIMARIA'],
  'countries': ['ESPANHA'],
  'issns': ['1578-1275', '0212-6567']}]

In [10]:
scielo = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/scielo.csv', 'scielo', debug=True)
scielo[0:3]

DEBUG. Field issns "ISSN SciELO" contains 9 chars.
DEBUG. Field issn_list contains 9 chars.
DEBUG. Field issn_list contains 19 chars.
DEBUG. Field issn_list contains 20 chars.
DEBUG. Field issn_list contains 21 chars.


[{'titles': ['ASOCIACION QUIMICA ARGENTINA',
   'J ARGENT CHEM SOC',
   'THE JOURNAL OF ARGENTINE CHEMICAL SOCIETY'],
  'countries': ['ARG'],
  'issns': ['1852-1428']},
 {'titles': ['ANALES DE LA ASOCIACION QUIMICA ARGENTINA',
   'AN ASOC QUIM ARGENT',
   'ASOCIACION QUIMICA ARGENTINA'],
  'countries': ['ARG'],
  'issns': ['0365-0375']},
 {'titles': ['ASOCIACION ARGENTINA DE NEUROCIRUGIA',
   'REV ARGENT NEUROCIR',
   'REVISTA ARGENTINA DE NEUROCIRUGIA'],
  'countries': ['ARG'],
  'issns': ['1850-1532']}]

In [11]:
scimagojr = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/scimagojr.csv', 'scimagojr', debug=True)
scimagojr[0:3]

DEBUG. Field issn_list contains 18 chars.


[{'titles': ['CA A CANCER JOURNAL FOR CLINICIANS'],
  'countries': ['UNITED STATES'],
  'issns': ['1542-4863', '0007-9235']},
 {'titles': ['FOUNDATIONS AND TRENDS IN MACHINE LEARNING'],
  'countries': ['UNITED STATES'],
  'issns': ['1935-8237', '1935-8245']},
 {'titles': ['NATURE REVIEWS MOLECULAR CELL BIOLOGY'],
  'countries': ['UNITED KINGDOM'],
  'issns': ['1471-0080', '1471-0072']}]

In [12]:
scopus_sources = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/scopus_sources.csv', 'scopus_sources', debug=True)
scopus_sources[0:3]

DEBUG. Field issns Print-ISSN contains 8 chars.
DEBUG. Field issns E-ISSN contains 8 chars.


[{'titles': ['@GRH'], 'countries': [], 'issns': ['2295-9149', '2034-9130']},
 {'titles': ['HIFU SKIN RESEARCH', 'SKIN RESEARCH'],
  'countries': [],
  'issns': ['0018-1390']},
 {'titles': ['MEDLINE UNIQUE TITLE',
   'NIPPON KOSHU EISEI ZASSHI JAPANESE JOURNAL OF PUBLIC HEALTH'],
  'countries': [],
  'issns': ['0546-1766']}]

In [13]:
scopus_accepted = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/scopus_accepted.csv', 'scopus_accepted', debug=True)
scopus_accepted[0:3]

DEBUG. Field issns EISSN contains 8 chars.
DEBUG. Field issns Print ISSN contains 8 chars.


[{'titles': ['INTERNATIONAL JOURNAL OF THE CARDIOVASCULAR ACADEMY'],
  'countries': [],
  'issns': ['2405-819X']},
 {'titles': ['CELS'], 'countries': [], 'issns': ['1407-7841', '2592-9356']},
 {'titles': ['STUDIA FENNICA HISTORICA'],
  'countries': [],
  'issns': ['1458-526X']}]

In [14]:
ulrich = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/ulrich.csv', 'ulrich', debug=True)
ulrich[0:3]

DEBUG. Field issns "title_issn" contains 9 chars.


[{'titles': ['RAHBURDHA YI MUDIRIYYAT DAR NIZAM I SALAMAT'],
  'countries': ['IRAN ISLAMIC REPUBLIC OF'],
  'issns': ['2476-6879']},
 {'titles': ['GLAUCUS'],
  'countries': ['UNITED KINGDOM'],
  'issns': ['0963-9519']},
 {'titles': ['D OLLE GRIEZE'],
  'countries': ['NETHERLANDS'],
  'issns': ['2588-8412']}]

In [15]:
wos_extra = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/wos_extra.csv', 'wos_extra', debug=True)
wos_extra[0:3]

DEBUG. Field issn_list contains 21 chars.
DEBUG. Field issns "uISSN" contains 9 chars.


[{'titles': ['STUDIA PRAWNICZE KUL'],
  'countries': ['POLAND'],
  'issns': ['2719-4264', '1897-7146']},
 {'titles': ['LEJEUNIA'], 'countries': ['BELGIUM'], 'issns': ['0457-4184']},
 {'titles': ['AMERICAN HEART JOURNAL PLUS CARDIOLOGY RESEARCH AND PRACTICE'],
  'countries': ['UNITED STATES OF AMERICA'],
  'issns': ['2666-6022']}]

In [16]:
wos_jcr = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/wos_jcr.csv', 'wos_jcr', debug=True)
wos_jcr[0:3]

DEBUG. Field issns eISSN contains 9 chars.
DEBUG. Field issns ISSN contains 9 chars.


[{'titles': ['2D MATERIALS', '2D MATER'],
  'countries': ['ENGLAND'],
  'issns': ['2053-1583']},
 {'titles': ['3 BIOTECH'],
  'countries': ['GERMANY'],
  'issns': ['2190-572X', '2190-5738']},
 {'titles': ['3D PRINT ADDIT MANUF', '3D PRINTING AND ADDITIVE MANUFACTURING'],
  'countries': ['UNITED STATES'],
  'issns': ['2329-7670', '2329-7662']}]

In [17]:
mi_extra = load_base('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/mi_extra.seq', name='mi_extra', debug=True)
mi_extra[0:3]

DEBUG. Field issns issn contains 9 chars.


[{'titles': ['ESCOLA ANNA NERY', 'ESC ANNA NERY REV ENFERM'],
  'countries': [],
  'issns': ['1414-8145']},
 {'titles': ['ESC ANNA NERY R ENFERM', 'ESCOLA ANNA NERY'],
  'countries': [],
  'issns': ['1414-8145']},
 {'titles': ['ESCOLA ANNA NERY', 'ESC ANNA NERY ONLINE'],
  'countries': [],
  'issns': ['1414-8145']}]

In [18]:
nlm = load_base_nlm('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/data/cleaned/nlm.txt')
nlm[0:3]

[{'titles': ['ACTA RADIOLOGICA THERAPY PHYSICS BIOLOGY'],
  'countries': ['SWEDEN'],
  'issns': ['0567-8064']},
 {'titles': ['ACTA PAEDIATRICA SCANDINAVICA'],
  'countries': ['SWEDEN'],
  'issns': ['0001-656X']},
 {'titles': ['ACTA MEDICA VETERINARIA',
   'TITLE ABBREVIATION ACTA MED VET NAPOLI'],
  'countries': ['ITALY'],
  'issns': ['0001-6136']}]

In [19]:
# Constrói os mapas 
#	ISSN-L --> {'issns': ...}
#	ISSN --> ISSN-L

issn_to_issnl = {}
issnl_to_data = {}

for line in open('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/issn_map.v0.8.csv'):
    gissn, issns = line.strip().split('|')
    issn_list = issns.split('#')

    for i in issn_list:
        if i not in issn_to_issnl:
            issn_to_issnl[i] = gissn

    if gissn not in issnl_to_data:
        issnl_to_data[gissn] = {
			'gold_issn': gissn,
			'issns': issns
		}

len(issn_to_issnl), len(issnl_to_data), issn_to_issnl['0000-0027'], issnl_to_data['0000-0027']

(2857813,
 2660709,
 '0000-0027',
 {'gold_issn': '0000-0027', 'issns': '0000-0027#2767-5505'})

In [20]:
def gen_sparse_values(data):
    for k in [
        'is_portal_issn_2019',
        'is_portal_issn_2024', 
        'is_doaj', 
        'is_latindex', 
        'is_ms_brazil', 
        'is_ms_spain', 
        'is_scielo', 
        'is_scimagojr', 
        'is_scopus_sources', 
        'is_scopus_accepted', 
        'is_ulrich', 
        'is_wos_extra', 
        'is_wos_jcr',
        'is_mi_extra',
        'is_nlm',
    ]:
        if k not in data:
            data[k] = "0"

In [21]:
# Alimenta o mapa ISSN-L --> códigos ISSN com títulos e países
#  ISSN-L --> {gold_issn': ..., 'issns': ..., 'titles': ..., 'countries': ...,}

for nb in [ 
    ('doaj', doaj), 
    ('latindex', latindex), 
    ('ms_brazil', ms_brazil), 
    ('ms_spain', ms_spain), 
    ('scielo', scielo), 
    ('scimagojr', scimagojr), 
    ('scopus_sources', scopus_sources), 
    ('scopus_accepted', scopus_accepted), 
    ('ulrich', ulrich),
    ('wos_extra', wos_extra), 
    ('wos_jcr', wos_jcr), 
    ('mi_extra', mi_extra), 
    ('nlm', nlm),
    ('portal_issn_2024', portal_issn_2024),
    ('portal_issn_2019', portal_issn_2019),
]:
    name, content = nb

    print(f'Carregando base {name}...')

    for row in content:
        r_titles = row['titles']
        r_countries = row['countries']
        r_issns = row['issns']

        if len(r_issns) == 0:
            continue

        gis = set()
        for i in r_issns:
            gis.add(issn_to_issnl[i])

        if len(gis) != 1:
            print(f'Houston we have a problem with {gis}')

        gissn = gis.pop()

        issnl_to_data[gissn][f'is_{name}'] = "1"

        if name == 'wos_jcr':
            issnl_to_data[gissn]['wos_jcr_countries'] = set(issnl_to_data[gissn].get('wos_jcr_countries', [])).union(set(r_countries))

        if name == 'wos_extra':
            issnl_to_data[gissn]['wos_extra_countries'] = set(issnl_to_data[gissn].get('wos_extra_countries', [])).union(set(r_countries))

        if name == 'scielo':
            issnl_to_data[gissn]['scielo_countries'] = set(issnl_to_data[gissn].get('scielo_countries', [])).union(set(r_countries))

        if name == 'latindex':
            issnl_to_data[gissn]['latindex_countries'] = set(issnl_to_data[gissn].get('latindex_countries', [])).union(set(r_countries))

        if name == 'portal_issn_2019':
            issnl_to_data[gissn]['portal_issn_2019_countries'] = set(issnl_to_data[gissn].get('portal_issn_2019_countries', [])).union(set(r_countries))
            issnl_to_data[gissn]['portal_issn_2019_main_title'] = set(issnl_to_data[gissn].get('portal_issn_2019_main_title', [])).union(row['main_title'])
            issnl_to_data[gissn]['portal_issn_2019_main_abbreviated_title'] = set(issnl_to_data[gissn].get('portal_issn_2019_main_abbreviated_title', [])).union(row['main_abbreviated_title'])

        if name == 'portal_issn_2024':
            issnl_to_data[gissn]['portal_issn_2024_countries'] = set(issnl_to_data[gissn].get('portal_issn_2024_countries', [])).union(set(r_countries))
            if issnl_to_data[gissn].get('portal_issn_2024_main_title', '') == '':
                issnl_to_data[gissn]['portal_issn_2024_main_title'] = row['main_title']

        if 'titles' not in issnl_to_data[gissn]:
            issnl_to_data[gissn]['titles'] = set()
        issnl_to_data[gissn]['titles'] = issnl_to_data[gissn]['titles'].union(set(r_titles))

        if 'countries' not in issnl_to_data[gissn]:
            issnl_to_data[gissn]['countries'] = set()
        issnl_to_data[gissn]['countries'] = issnl_to_data[gissn]['countries'].union(set(r_countries))

        gen_sparse_values(issnl_to_data[gissn])

Carregando base doaj...
Carregando base latindex...
Carregando base ms_brazil...
Carregando base ms_spain...
Carregando base scielo...
Carregando base scimagojr...
Carregando base scopus_sources...
Carregando base scopus_accepted...
Carregando base ulrich...
Carregando base wos_extra...
Carregando base wos_jcr...
Carregando base mi_extra...
Carregando base nlm...
Carregando base portal_issn_2024...
Carregando base portal_issn_2019...


In [22]:
for i in issnl_to_data:
	for k in [
		'wos_jcr_countries', 
		'wos_extra_countries', 
		'scielo_countries', 
		'latindex_countries', 
		'portal_issn_2019_countries', 
		'portal_issn_2019_main_title', 
		'portal_issn_2019_main_abbreviated_title', 
		'portal_issn_2024_countries', 
		'portal_issn_2024_main_title',
		'titles',
		'countries',
	]:
		v = issnl_to_data[i].get(k)
		if isinstance(v, set):
			issnl_to_data[i][k] = '#'.join(v)

In [23]:
issnl_to_data[issn_to_issnl['0000-0019']]

{'gold_issn': '0000-0019',
 'issns': '0000-0019#2150-4008',
 'is_ulrich': '1',
 'titles': 'PW PUBLISHERS WEEKLY#THE PUBLISHERS WEEKLY#PW#PUBLISHERS WEEKLY#PUBL WKLY#PWK#PUBLISHERS WEEKLY ONLINE#PUBLISHERS WEEKLY THE INTERNATIONAL NEWS MAGAZINE OF BOOK PUBLISHING',
 'countries': 'NEW JERSEY#UNITED STATES#CALIFORNIA',
 'is_portal_issn_2019': '1',
 'is_portal_issn_2024': '1',
 'is_doaj': '0',
 'is_latindex': '0',
 'is_ms_brazil': '0',
 'is_ms_spain': '0',
 'is_scielo': '0',
 'is_scimagojr': '0',
 'is_scopus_sources': '0',
 'is_scopus_accepted': '0',
 'is_wos_extra': '0',
 'is_wos_jcr': '0',
 'is_mi_extra': '0',
 'is_nlm': '0',
 'portal_issn_2024_countries': 'NEW JERSEY#CALIFORNIA',
 'portal_issn_2024_main_title': 'THE PUBLISHERS WEEKLY',
 'portal_issn_2019_countries': 'UNITED STATES',
 'portal_issn_2019_main_title': 'THE PUBLISHERS WEEKLY',
 'portal_issn_2019_main_abbreviated_title': 'PUBL WKLY'}

In [24]:
header = [
    'gold_issn',
    'issns',
    'portal_issn_2019_main_title',
    'portal_issn_2019_main_abbreviated_title',
    'portal_issn_2024_main_title',
    'titles',
    'countries',
    'wos_jcr_countries',
    'wos_extra_countries',
    'scielo_countries',
    'latindex_countries',
    'portal_issn_2024_countries',
    'portal_issn_2019_countries',
    'is_doaj',
    'is_latindex',
    'is_ms_brazil',
    'is_ms_spain',
    'is_mi_extra',
    'is_nlm',
    'is_portal_issn_2019',
    'is_portal_issn_2024',
    'is_scielo',
    'is_scimagojr',
    'is_scopus_accepted',
    'is_scopus_sources',
    'is_ulrich',
    'is_wos_extra',
    'is_wos_jcr'
]

title_keys = ['portal_issn_2019_main_title', 'portal_issn_2019_main_abbreviated_title', 'portal_issn_2024_main_title', 'titles']

invalid_data = []

with open('/home/rafaeljpd/Data/cimetrias/correction-bases/v0.8/issn_to_all.v0.8.csv', 'w') as fout:
    fout.write('|'.join(header).upper() + '\n')
    for i in issnl_to_data:
        is_valid = 0

        for tk in title_keys:
            if issnl_to_data[i].get(tk, '') != '':
                is_valid += 1
        
        if is_valid > 0:
            fout.write('|'.join([issnl_to_data[i].get(k, '') for k in header]) + '\n')
        else:
            invalid_data.append(issnl_to_data[i])

In [25]:
invalid_data

[{'gold_issn': '0000-0035',
  'issns': '0000-0035',
  'is_portal_issn_2019': '1',
  'portal_issn_2019_countries': '',
  'portal_issn_2019_main_title': '',
  'portal_issn_2019_main_abbreviated_title': '',
  'titles': '',
  'countries': '',
  'is_portal_issn_2024': '0',
  'is_doaj': '0',
  'is_latindex': '0',
  'is_ms_brazil': '0',
  'is_ms_spain': '0',
  'is_scielo': '0',
  'is_scimagojr': '0',
  'is_scopus_sources': '0',
  'is_scopus_accepted': '0',
  'is_ulrich': '0',
  'is_wos_extra': '0',
  'is_wos_jcr': '0',
  'is_mi_extra': '0',
  'is_nlm': '0'},
 {'gold_issn': '0000-0086',
  'issns': '0000-0086',
  'is_portal_issn_2019': '1',
  'portal_issn_2019_countries': '',
  'portal_issn_2019_main_title': '',
  'portal_issn_2019_main_abbreviated_title': '',
  'titles': '',
  'countries': '',
  'is_portal_issn_2024': '0',
  'is_doaj': '0',
  'is_latindex': '0',
  'is_ms_brazil': '0',
  'is_ms_spain': '0',
  'is_scielo': '0',
  'is_scimagojr': '0',
  'is_scopus_sources': '0',
  'is_scopus_acce